In [7]:
import sys
sys.path.insert(0, '../')
import pandas as pd
import numpy as np
import time
from datetime import datetime
import matplotlib as mpl
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import funcs
#import pvlib
import os
import glob
from flats import Building
from flats import Flat
from flats import PV_system

datenote = '2026-01-20'

In [ ]:
# Plot the individual flat data
b_path = '2026-01-20_run_10_2025-06-27_Buildings_100.xlsx' # Building info file
ind_path = './Test_Magda_unshaded\Apartment/2026-01-20_apartment_Unconstrained_' # Common path for individual files
system = 'south_MPV_12_VBPV_large' # System name
df_coll = pd.DataFrame()

# Color dict for different flat types
plot_color = {'Studio': 'b',
              '1-room': 'g',
              '2-room-S': 'm',
              '2-room-L': 'r',
              '3-room': 'c'}

# Marker dict for different electricity purchase contracts
plot_marker = {'Fixed': 'o',
               'Spot': 's',
               'UFN': 'D',}

# Loop through the buildings
for i in range(10):
    building_info = pd.read_excel(b_path, sheet_name=f'Building_{i+1}', index_col=0)
    building_info.index = building_info.index.map(lambda x: f'Building_{i+1}_' + str(x)) # Add building number to index

    trade_info = pd.read_excel(ind_path + f'Building_{i+1}_mix.xlsx', sheet_name=system + '_trade', index_col=0)
    trade_info.index = trade_info.index.map(lambda x: f'Building_{i+1}_' + str(x)) # Add building number to index
    comb_info = pd.concat([building_info, trade_info], axis=1)
    df_coll = pd.concat([df_coll, comb_info], axis=0)

df_coll['PV_volume'] = df_coll['PV_bought'] + df_coll['PV_sold']
df_coll['value_volume'] = df_coll['value_bought'] + df_coll['value_sold']
df_coll['PV_balance'] = df_coll['PV_sold'] - df_coll['PV_bought']
df_coll['value_balance'] = df_coll['value_sold'] - df_coll['value_bought']

fig, axs = plt.subplots(1,3, figsize=(18, 5))

# Set the labels and limits for the plots
axs[0].set_xlabel('PV trade volume (kWh)')
axs[0].set_ylabel('PV trade balance (kWh)')
axs[1].set_xlabel('Value trade volume (EUR)')
axs[1].set_ylabel('Value trade balance (EUR)')
axs[2].set_xlabel('PV trade balance (kWh)')
axs[2].set_ylabel('Value trade balance (EUR)')
axs[0].grid()
axs[0].set_xlim([50, 500])
axs[0].set_ylim([-300, 200])
axs[0].set_yticks(np.arange(-300, 250, 50))
axs[1].set_xlim([0, 30])
axs[1].set_ylim([-25, 10])
axs[1].grid()
axs[2].grid()
axs[2].set_xlim([-300, 200])
axs[2].set_ylim([-25, 10])
custom_handles, custom_labels = [], []
for item in df_coll.index:
    if not item.__contains__('Checksum'): # Don't plot the checksum rows
        
        # Plot the data with correct color and marker
        handle = axs[0].plot(df_coll.loc[item, 'PV_volume'], df_coll.loc[item, 'PV_balance'], label=item, linestyle='None', color=plot_color[df_coll.loc[item, 'Flat_type']], marker=plot_marker[df_coll.loc[item, 'Default_contract']])
        axs[1].plot(df_coll.loc[item, 'value_volume'], df_coll.loc[item, 'value_balance'], label=item, color=plot_color[df_coll.loc[item, 'Flat_type']], marker=plot_marker[df_coll.loc[item, 'Default_contract']])
        axs[2].plot(df_coll.loc[item, 'PV_balance'], df_coll.loc[item, 'value_balance'], label=item, color=plot_color[df_coll.loc[item, 'Flat_type']], marker=plot_marker[df_coll.loc[item, 'Default_contract']])

        # Add custom handles and labels for the legend
        if (df_coll.loc[item, 'Flat_type'] not in custom_labels) and (df_coll.loc[item, 'Default_contract'] == 'Fixed'):
            custom_handles.append(handle[0])
            custom_labels.append(df_coll.loc[item, 'Flat_type'])

axs[0].legend(custom_handles, custom_labels, loc='lower left', bbox_to_anchor=(0, 0))

# Add the text labels to the plots
axs[0].text(-0.15, 0.98, 'd', transform=axs[0].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[1].text(-0.15, 0.98, 'e', transform=axs[1].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[2].text(-0.15, 0.98, 'f', transform=axs[2].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))

# Save
fig.savefig(datenote + f'_PV_trade_balance_{system}.jpeg', dpi=300, bbox_inches='tight')
df_coll.to_excel(datenote + f'_PV_trade_balance_{system}.xlsx')

In [ ]:
# Plot the individual flat data for the electricity bill

b_path = './Work_folder/2025-07-29_2_2025-06-27_Buildings_10.xlsx' # Building info file
ind_path = './Work_folder/2025-07-29_2_unshaded_10_Unconstrained_' # Common path for individual files
systems = ['south_MPV_12_MPV_facade', 'south_MPV_12_VBPV_large'] # System name
df_coll = pd.DataFrame()

# Color dict for different flat types
plot_color = {'Studio': 'b',
              '1-room': 'g',
              '2-room-S': 'm',
              '2-room-L': 'r',
              '3-room': 'c'}

# Marker dict for different electricity purchase contracts
plot_marker = {'Fixed': 'o',
               'Spot': 's',
               'UFN': 'D',}

fig, axs = plt.subplots(1,3, figsize=(18,5))
j=0

for system in systems:
# Loop through the buildings
    for i in range(10):
        building_info = pd.read_excel(b_path, sheet_name=f'Building_{i+1}', index_col=0)
        building_info.index = building_info.index.map(lambda x: f'Building_{i+1}_' + system + '_' + str(x)) # Add building number to index

        trade_info = pd.read_excel(ind_path + f'Building_{i+1}_mix.xlsx', sheet_name=system + '_trade', index_col=0)
        trade_info.index = trade_info.index.map(lambda x: f'Building_{i+1}_' + system + '_' + str(x)) # Add building number to index
        comb_info = pd.concat([building_info, trade_info], axis=1)
        df_coll = pd.concat([df_coll, comb_info], axis=0)

custom_handles, custom_labels = [], []
custom_handles2, custom_labels2 = [], []

for item in df_coll.index:
    if not item.__contains__('Checksum'): # Don't plot the checksum rows
                    
        if item.__contains__('MPV_facade'):
            j = 0
        elif item.__contains__('VBPV_large'):
            j = 1
        # Plot the data with correct color and marker
        handle = axs[j].plot(df_coll.loc[item, 'El_cost_no_pv'], df_coll.loc[item, 'El_cost_no_pv'] - df_coll.loc[item, 'El_bill'], label=item, linestyle='None', \
                                color=plot_color[df_coll.loc[item, 'Flat_type']], marker=plot_marker[df_coll.loc[item, 'Default_contract']])
        
        if j==1:
            mface = 'none'
        else:
            mface = plot_color[df_coll.loc[item, 'Flat_type']]
        
        handle2 = axs[2].plot(df_coll.loc[item, 'El_cost_no_pv'], 100*(df_coll.loc[item, 'El_cost_no_pv'] - df_coll.loc[item, 'El_bill']) / df_coll.loc[item, 'El_cost_no_pv'], label=item, linestyle='None', \
                                color=plot_color[df_coll.loc[item, 'Flat_type']], marker=plot_marker[df_coll.loc[item, 'Default_contract']], markerfacecolor=mface)

        # Add custom handles and labels for the legend
        if (df_coll.loc[item, 'Flat_type'] not in custom_labels) and (df_coll.loc[item, 'Default_contract'] == 'Fixed'):
            custom_handles.append(handle[0])
            custom_labels.append(df_coll.loc[item, 'Flat_type'])

        if (item.__contains__('MPV_facade')) and '$\\mathrm{MPV_F}$' not in custom_labels2:
            custom_handles2.append(handle2[0])
            custom_labels2.append('$\\mathrm{MPV_F}$')
        elif (item.__contains__('VBPV_large')) and '$\\mathrm{VBPV_L}$' not in custom_labels2:
            custom_handles2.append(handle2[0])
            custom_labels2.append('$\\mathrm{VBPV_L}$')

# Set the labels and limits for the plots
axs[0].set_xlabel('Electricity bill without PV (EUR)')
axs[0].set_ylabel('PV value (EUR)')
axs[1].set_xlabel('Electricity bill without PV (EUR)')
axs[1].set_ylabel('PV value (EUR)')
axs[2].set_xlabel('Electricity bill without PV (EUR)')
axs[2].set_ylabel('Relative savings with PV (%)')
axs[0].grid()
axs[0].set_xlim([200, 900])
axs[0].set_ylim([0, 250])
#axs[0].set_yticks(np.arange(-300, 250, 50))
axs[1].set_xlim([200, 900])
axs[1].set_ylim([0, 250])
axs[1].grid()

axs[2].set_xlim([200, 900])
axs[2].set_ylim([14, 30])
axs[2].grid()

axs[0].legend(custom_handles, custom_labels, loc='upper left', bbox_to_anchor=(0, 1))
axs[2].legend(custom_handles2, custom_labels2, loc='lower right', bbox_to_anchor=(1, 0))

# Add the text labels to the plots
axs[0].text(-0.15, 0.98, 'g', transform=axs[0].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[1].text(-0.15, 0.98, 'h', transform=axs[1].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[2].text(-0.15, 0.98, 'i', transform=axs[2].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))

# Save
fig.savefig(datenote + f'_El_bill_PV_value.jpeg', dpi=300, bbox_inches='tight')
df_coll.to_excel(datenote + f'_El_bill_PV_value.xlsx')

In [ ]:
# Plot the individual flat data for the electricity bill
# Last figure in the manuscript

b_path = '2026-01-20_run_10_2025-06-27_Buildings_100.xlsx' # Building info file
ind_path_U = './Test_Magda_unshaded/Apartment/Fixed/2026-01-20_apartment_Unconstrained_' # Common path for individual files
ind_path_D = './Test_Magda_unshaded/Apartment/Fixed/2026-01-20_apartment_Disabled_' # Common path for individual files
systems = ['south_MPV_12_MPV_facade', 'south_MPV_12_VBPV_large'] # System name
df_coll_U = pd.DataFrame()
df_coll_D = pd.DataFrame()

roof_kWp = {'MPV_12_monofacial': 5.28, 'MPV_45_monofacial': 19.8} # kWp for roof PV systems
facade_kWp = {'south_MPV_12_MPV_facade': 9.24, 'south_MPV_12_VBPV_large': 9.24, '2025-03-21_VBPV_small_F_bifacial': 4.582, '2025-03-21_VBPV_hybrid_F_bifacial': 7.668} # kWp for facade PV systems
plt.rcParams.update({'font.size': 12})

# Color dict for different flat types
plot_color = {'Studio': 'b',
              '1-room': 'g',
              '2-room-S': 'm',
              '2-room-L': 'r',
              '3-room': 'c'}

# Marker dict for different electricity purchase contracts
plot_marker = {'Fixed': 'o',
               'Spot': 's',
               'UFN': 'D'}

rel_areas = {'Studio': 34/1026,
               '1-room': 47/1026,
               '2-room-S': 65/1026,
                '2-room-L': 74.5/1026,
                '3-room': 86.5/1026} # Relative areas of the flats

fig, axs = plt.subplots(2,2, figsize=(12,10))
fig2, axs2 = plt.subplots(1,2, figsize=(12,5))
j=0

for system in systems:
# Loop through the buildings
    for i in range(10):
        building_info = pd.read_excel(b_path, sheet_name=f'Building_{i+1}', index_col=0)
        building_info.index = building_info.index.map(lambda x: f'Building_{i+1}_' + system + '_' + str(x)) # Add building number to index

        # Download the trade info
        trade_info = pd.read_excel(ind_path_U + f'Building_{i+1}_mix.xlsx', sheet_name=system, index_col=0)
        trade_info.index = trade_info.index.map(lambda x: f'Building_{i+1}_' + system + '_' + str(x)) # Add building number to index
        comb_info_U = pd.concat([building_info, trade_info], axis=1)
        df_coll_U = pd.concat([df_coll_U, comb_info_U], axis=0)

        flat_info = pd.read_excel(ind_path_D + f'Building_{i+1}_mix.xlsx', sheet_name=system, index_col=0)
        flat_info.index = flat_info.index.map(lambda x: f'Building_{i+1}_' + system + '_' + str(x)) # Add building number to index
        comb_info_D = pd.concat([building_info, flat_info], axis=1)
        df_coll_D = pd.concat([df_coll_D, comb_info_D], axis=0)

# For legends
custom_handles, custom_labels = [], []
custom_handles2, custom_labels2 = [], []

# Calculate additional columns for the plot
df_coll_U['PV_volume'] = df_coll_U['PV_bought'] + df_coll_U['PV_sold']
df_coll_U['value_volume'] = df_coll_U['value_bought'] + df_coll_U['value_sold']
df_coll_U['PV_balance'] = df_coll_U['PV_sold'] - df_coll_U['PV_bought']
df_coll_U['value_balance'] = df_coll_U['value_sold'] - df_coll_U['value_bought']

for item in df_coll_U.index:
    if not item.__contains__('Checksum'): # Don't plot the checksum rows

        # Set marker face color based on the system type            
        if item.__contains__('MPV_facade'):
            mface = plot_color[df_coll_U.loc[item, 'Flat_type']]
        elif item.__contains__('VBPV_large'):
            mface = 'none'
        # Plot the data with correct color and marker

        kWp = roof_kWp['MPV_12_monofacial'] + facade_kWp[system]
        rel_area = rel_areas[df_coll_U.loc[item, 'Flat_type']]

        handle2_1 = axs2[0].plot(df_coll_U.loc[item, 'PV_volume'], df_coll_U.loc[item, 'PV_balance'], label=item, linestyle='None', color=plot_color[df_coll_U.loc[item, 'Flat_type']], \
                                marker=plot_marker[df_coll_U.loc[item, 'Default_contract']], markerfacecolor=mface)
        handle2_2 = axs2[1].plot(df_coll_U.loc[item, 'value_volume'], df_coll_U.loc[item, 'value_balance'], label=item, color=plot_color[df_coll_U.loc[item, 'Flat_type']], \
                      marker=plot_marker[df_coll_U.loc[item, 'Default_contract']], markerfacecolor=mface)

        handle = axs[0,0].plot(df_coll_U.loc[item, 'El_cost_no_pv'], (df_coll_U.loc[item, 'El_cost_no_pv'] - df_coll_U.loc[item, 'El_bill']) / (rel_area * kWp), label=item, linestyle='None', \
                                color=plot_color[df_coll_U.loc[item, 'Flat_type']], marker=plot_marker[df_coll_U.loc[item, 'Default_contract']], markerfacecolor=mface)
        
        handle2 = axs[1,0].plot(df_coll_U.loc[item, 'El_cost_no_pv'], 100*(df_coll_U.loc[item, 'El_cost_no_pv'] - df_coll_U.loc[item, 'El_bill']) / df_coll_U.loc[item, 'El_cost_no_pv'], label=item, linestyle='None', \
                                color=plot_color[df_coll_U.loc[item, 'Flat_type']], marker=plot_marker[df_coll_U.loc[item, 'Default_contract']], markerfacecolor=mface)

        # Add custom handles and labels for the legend
        if (df_coll_U.loc[item, 'Flat_type'] not in custom_labels) and (df_coll_U.loc[item, 'Default_contract'] == 'Fixed'):
            custom_handles.append(handle[0])
            custom_labels.append(df_coll_U.loc[item, 'Flat_type'])

        if (item.__contains__('MPV_facade')) and '$\\mathrm{MPV_F}$' not in custom_labels2:
            custom_handles2.append(handle2[0])
            custom_labels2.append('$\\mathrm{MPV_F}$')
            
        elif (item.__contains__('VBPV_large')) and '$\\mathrm{VBPV_L}$' not in custom_labels2:
            custom_handles2.append(handle2[0])
            custom_labels2.append('$\\mathrm{VBPV_L}$')

for item in df_coll_D.index:
    if not item.__contains__('Checksum'): # Don't plot the checksum rows
        
        # Set marker face color based on the system type            
        if item.__contains__('MPV_facade'):
            mface = plot_color[df_coll_D.loc[item, 'Flat_type']]
        elif item.__contains__('VBPV_large'):
            mface = 'none'
        # Plot the data with correct color and marker
        kWp = roof_kWp['MPV_12_monofacial'] + facade_kWp[system]
        rel_area = rel_areas[df_coll_D.loc[item, 'Flat_type']]

        axs[0,1].plot(df_coll_D.loc[item, 'El_cost_no_pv'], (df_coll_D.loc[item, 'El_cost_no_pv'] - df_coll_D.loc[item, 'El_bill']) / (rel_area * kWp), label=item, linestyle='None', \
                                color=plot_color[df_coll_D.loc[item, 'Flat_type']], marker=plot_marker[df_coll_D.loc[item, 'Default_contract']], markerfacecolor=mface)
        
        axs[1,1].plot(df_coll_D.loc[item, 'El_cost_no_pv'], 100*(df_coll_D.loc[item, 'El_cost_no_pv'] - df_coll_D.loc[item, 'El_bill']) / df_coll_D.loc[item, 'El_cost_no_pv'], label=item, linestyle='None', \
                                color=plot_color[df_coll_D.loc[item, 'Flat_type']], marker=plot_marker[df_coll_D.loc[item, 'Default_contract']], markerfacecolor=mface)

# Set the labels and limits for the plots
axs2[0].set_xlabel('PV trade volume (kWh)', fontsize=12)
axs2[0].set_ylabel('PV trade balance (kWh)', fontsize=12)
axs2[1].set_xlabel('Value trade volume (EUR)', fontsize=12)
axs2[1].set_ylabel('Value trade balance (EUR)', fontsize=12)
axs2[0].grid()
axs2[0].set_xlim([50, 450])
axs2[0].set_ylim([-300, 200])
axs2[1].set_xlim([0, 25])
axs2[1].set_ylim([-20, 10])
axs2[1].grid()

axs[0,0].set_xlabel('Electricity bill without PV (EUR)', fontsize=12)
axs[0,0].set_ylabel('PV value (EUR/kW)', fontsize=12)
axs[1,0].set_xlabel('Electricity bill without PV (EUR)', fontsize=12)
axs[1,0].set_ylabel('Relative savings with PV (%)', fontsize=12)

axs[0,1].set_xlabel('Electricity bill without PV (EUR)', fontsize=12)
axs[0,1].set_ylabel('PV value (EUR/kW)', fontsize=12)
axs[1,1].set_xlabel('Electricity bill without PV (EUR)', fontsize=12)
axs[1,1].set_ylabel('Relative savings with PV (%)', fontsize=12)

axs[0,0].grid()
axs[0,1].grid()
axs[1,0].grid()
axs[1,1].grid()

axs[0,0].set_xlim([200, 900])
axs[0,0].set_ylim([80, 180])
#axs[0].set_yticks(np.arange(-300, 250, 50))
axs[0,1].set_xlim([200, 900])
axs[0,1].set_ylim([80, 180])

axs[1,0].set_xlim([200, 900])
axs[1,0].set_ylim([14, 30])
#axs[0].set_yticks(np.arange(-300, 250, 50))

axs[1,1].set_xlim([200, 900])
axs[1,1].set_ylim([14, 30])

axs[0,1].legend(custom_handles, custom_labels, loc='upper right', bbox_to_anchor=(1, 1), ncols=2, fontsize=12)
axs[1,1].legend(custom_handles2, custom_labels2, loc='upper right', bbox_to_anchor=(1, 1), fontsize=12)
axs2[0].legend(custom_handles, custom_labels, loc='lower left', bbox_to_anchor=(0, 0), ncols=2, fontsize=12)
axs2[1].legend(custom_handles2, custom_labels2, loc='lower left', bbox_to_anchor=(0, 0), fontsize=12)

# Add the text labels to the plots
axs[0,0].text(-0.15, 0.98, 'a', transform=axs[0,0].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[0,1].text(-0.15, 0.98, 'b', transform=axs[0,1].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[1,0].text(-0.15, 0.98, 'c', transform=axs[1,0].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs[1,1].text(-0.15, 0.98, 'd', transform=axs[1,1].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs2[0].text(-0.15, 0.98, 'a', transform=axs2[0].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))
axs2[1].text(-0.15, 0.98, 'b', transform=axs2[1].transAxes, fontsize=12, va='top', ha='left', bbox=dict(facecolor='white', edgecolor='none', pad=2.0))

# Save
fig.savefig(datenote + f'_El_bill_PV_value_comb.jpeg', dpi=300, bbox_inches='tight')
fig2.savefig(datenote + f'_PV_trade.jpeg', dpi=300, bbox_inches='tight')
df_coll_U.to_excel(datenote + f'_El_bill_PV_value_comb_U.xlsx')
df_coll_D.to_excel(datenote + f'_El_bill_PV_value_comb_D.xlsx')

In [ ]:
# Calculate the average saving percentage in the electricity costs for each flat type and system

average_pros = pd.DataFrame()
print(df_coll_D['El_cost_no_pv'])
df_coll_D['Percentage'] = 100*(df_coll_D['El_cost_no_pv'] - df_coll_D['El_bill']) / df_coll_D['El_cost_no_pv']
df_coll_U['Percentage'] = 100*(df_coll_U['El_cost_no_pv'] - df_coll_U['El_bill']) / df_coll_U['El_cost_no_pv']

average_pros['U'] = df_coll_U.groupby('Flat_type')['Percentage'].mean()
average_pros['D'] = df_coll_D.groupby('Flat_type')['Percentage'].mean()

print(average_pros)
average_pros.to_excel(datenote + f'_average_pros.xlsx')

In [ ]:
# Plots monthly profiles and annual series for the data given as input

def ave_data_monthly(
        data_in : pd.DataFrame
        ) -> pd.DataFrame:
    """
    Calculates average monthly profiles with hourly resolution
    INPUT:
    data_in: input data, e.g., annual data set, DataFrame
    OUTPUT:
    data_out: monthly average profiles, DataFrame
    """
    ind = data_in.index  # Datetimes
    col = data_in.columns  # Variable names

    data_out = pd.DataFrame(index=np.arange(0, 12*24), columns=col)
    month_array = np.arange(1, 13, 1)
    hour_array = np.arange(0, 24, 1)
    # Goes through each month
    for j in month_array:
        data_month = data_in.loc[ind.month == j, :]
        
        for i in hour_array:
            data_hour = data_month.loc[data_month.index.hour == i, :]
            ind_out = (j - 1)*24 + i

            for k in col:
                data_out.loc[ind_out, k] = np.mean(data_hour.loc[:,k])

    return data_out

def monthly_plots(data):
    plt.rcParams.update({'font.size': 16})
    plt.rcParams.update({'mathtext.default': 'regular'})
    #fig,axs = plt.subplots(3,4, figsize=[30,20])
    fig,axs = plt.subplots(2,3, figsize=[20,15])
    data_plot = ave_data_monthly(data)
    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    
    #linstyl = ['m-','b-','b--','r-','r--','g-','g--','k-','k--','c-','c--']
    linstyl = ['b-','r-','g-','k-','c-']

    #for i in range(1, 13, 1):
    for i in range(4, 10, 1):
        ii = i - 3
        #coord = [int(np.floor((i-1)/4)),np.remainder((i-1),4)]
        coord = [int(np.floor((ii-1)/3)),np.remainder((ii-1),3)]
        
        for j in range(len(data_plot.columns)):
            axs[coord[0],coord[1]].plot(np.arange(0,24), data_plot.iloc[(i-1)*24:i*24,j])#), linstyl[j])
            
        axs[coord[0],coord[1]].set_xlabel('Hour')
        #axs[coord[0],coord[1]].set_ylabel('Electricity consumption (kWh/h)')
        #axs[coord[0],coord[1]].set_ylabel('Electricity production (kWh/(h*kW))')
        axs[coord[0],coord[1]].set_ylabel('Electricity price (c/kWh)')
        axs[coord[0],coord[1]].set_title(str(months[i-1]))        
        axs[coord[0],coord[1]].set_xlim([0,23])
        #axs[coord[0],coord[1]].set_ylim([0,24])
        axs[coord[0],coord[1]].set_xticks(np.arange(2,26,4))
        axs[coord[0],coord[1]].grid()
        if i == 6:
            axs[coord[0],coord[1]].legend(data.columns)
            #axs[coord[0],coord[1]].legend(['Real', 'Regression fit'])
    return fig

def annual_series(data):
    plt.rcParams.update({'font.size': 16})
    plt.rcParams.update({'mathtext.default': 'regular'})
    fig,axs = plt.subplots(12,1, figsize=[30,60])
    data_plot = data
    n_points = len(data_plot.index)
    k = int(n_points/12)  # Number of points per subplot
    
    #linstyl = ['m-','b-','b--','r-','r--','g-','g--','k-','k--','c-','c--']
    linstyl = ['b-','r-','g-','k-','c-']

    for i in range(12):
        tstamp = data_plot.index.month == (i+1)
        axs[i].plot(data_plot.loc[tstamp,:])#), linstyl[j])
        axs[i].set_xlabel('Date')
        #axs[i].set_ylabel('Electricity consumption (kWh/h)')
        axs[i].set_ylabel('Electricity production (kWh/(h*kW))')
        #axs[i].set_ylabel('Electricity price (c/kWh)')
        axs[i].grid()
        if i == 0:
            axs[i].legend(data.columns)
    return fig

#file = './Files/2025-02-24_Proper_run/Consumption/Profiles_filled/2025-04-04_Cleaned_Adv_el_heat_filtered_12MWh_profiles_filled.xlsx'
#file = './Files/2025-02-24_Proper_run/Consumption/Profiles_filled/2025-06-16_Profiles_Jerzy.xlsx'
file = './Files/Input_check/2026-01-20_Unshaded_check.xlsx'
#data = pd.read_excel(file,index_col=0,sheet_name='Sheet1')
data = pd.read_excel(file, index_col=0)
columns = ['158_small_panels_south_E_bifacial','large_facade_panels_south_E_bifacial','large_facade_panels_south_S_monofacial','MPV_12_panels_south_monofacial','small_and_large_panels_south_E_bifacial']
ind_all = data.index
#data = pd.read_csv(file)
#data.index = ind_all

fig = monthly_plots(data[columns])
fig2 = annual_series(data[columns])
fig.savefig(datenote+'_Average_prod_profiles.jpeg',dpi=300)
fig2.savefig(datenote+'_Full_prod_profiles.jpeg',dpi=300)